In [107]:
import pandas as pd
import numpy as np

In [108]:
df = pd.read_csv('raw_data.csv')

In [109]:
df.duplicated().sum()

np.int64(70)

In [110]:
df = df.drop_duplicates()

In [111]:
df.isnull().sum()

Event_ID                 0
Line                     0
Station                  0
Machine_ID               0
Tool_ID                  0
Operator_ID              0
TS                       0
Shift                    0
Part_No                  0
Part_Type                0
BOM_Version              0
Torque                   0
Temp                     0
Pressure                 0
Vibration                0
Humidity                 0
Voltage                  0
Current                  0
Energy_Consumption       0
Cycle_Time               0
Defect                1559
Defect_Code           2804
Scrap_Reason          3440
Inspection_Source        0
Inspection_Result      135
VIN                      0
Supplier                 0
Supplier_Lot             0
Batch_ID                 0
WO                       0
Production_Order         0
Material_Type            0
Assembly_Step            0
Calibration_Factor       0
Sensor_Status            0
Rework_Flag              0
Quality_Score            0
A

1. Line and Station

In [112]:
df['Line'] = df['Line'].astype(str).str.strip().str.upper()
df["Station"] = df["Station"].astype(str).str.strip().str.upper()

df['Line'] = df['Line'].str.replace(" ", "", regex=False)
df['Station'] = df['Station'].str.replace(" ", "", regex=False)

2. TimeStamp

In [113]:
df["TS"] = df["TS"].ffill()

df["TS"] = df["TS"].str.replace(" :", ":", regex=False)

time = df["TS"].str.split(" ").str[1]

hour = time.str.split(":").str[0].astype(int)
minute = time.str.split(":").str[1].astype(int)

hour = hour + (minute//60)
minute = minute % 60

df["TS"] = df["TS"].str.split(" ").str[0] + " " + hour.astype(str).str.zfill(2) + ":" + minute.astype(str).str.zfill(2)

df["TS"] = pd.to_datetime(df["TS"], errors="coerce", format="mixed",dayfirst=True)

In [114]:
df['TS'].isnull().sum()

np.int64(6)

In [115]:
df.loc[df["TS"].isna(), ["Event_ID","TS","Shift","Station","VIN"]]

,Event_ID,TS,Shift,Station,VIN
2115,2116,NaT,Morning,STN-08,AG01Z77EAU3Y06JVC
4946,4947,NaT,Night,STN-05,A22Y2VW15RR5XXJNX
4950,4951,NaT,Morning,STN-07,PJRLADD14K1TFSLXK
4995,4996,NaT,Evening,STN-09,0LAENPFMWADTXKD2V
4996,4997,NaT,Evening,STN-01,XUL8G5SJ2X2RNF120
5035,5036,NaT,Evening,STN-09,SHUCF6ACKLA0L5XTZ


In [116]:
shift_map = {
    "Morning": "06:00:00",
    "Evening": "14:00:00",
    "Night": "22:00:00"
}

mask = df["TS"].isna()

df.loc[mask, "TS"] = pd.to_datetime(
    "2026-03-10 " + df.loc[mask, "Shift"].map(shift_map)
)

3. Part number normalization and BOM validation.

In [117]:

df["Part_No"] = df["Part_No"].astype(str).str.strip().str.upper()

# fix missing hyphen
df["Part_No"] = df["Part_No"].str.replace(r'^(\d{3})([A-Z]{3})$', r'\1-\2', regex=True)

# validate format   
df["Part_Format_Valid"] = df["Part_No"].str.match(r'^\d{3}-[A-Z]{3}$')

In [118]:
bom_parts = df["Part_No"].unique() 
df["Part_In_BOM"] = df["Part_No"].isin(bom_parts)

4. Torque numeric parsing to Nm, unit checks.

In [119]:
# clean torque column
df["Torque"] = df["Torque"].astype(str).str.strip().str.upper()

# extract numeric value
torque_val = df["Torque"].str.extract(r'(\d+\.?\d*)')[0].astype(float)

# detect lb-ft values
mask = df["Torque"].str.contains("LB")

# convert lb-ft to Nm
torque_val.loc[mask] = torque_val.loc[mask] * 1.35582

# replace original column with cleaned Nm values
df["Torque"] = torque_val

5. Temperature unit parsing (C/F) to Celsius.

In [120]:
# clean temperature column
df["Temp"] = df["Temp"].astype(str).str.strip().str.upper()

# extract numeric value
temp_val = df["Temp"].str.extract(r'(\d+\.?\d*)')[0].astype(float)

# extract unit
temp_unit = df["Temp"].str.extract(r'([CF])')[0]

# convert to Celsius
df["Temp"] = temp_val
df.loc[temp_unit == "F", "Temp_C"] = (temp_val - 32) * 5/9

6. Defect field normalization (None/OK/Reject/Repair).


In [121]:
df["Defect"] = df["Defect"].astype(str).str.strip().str.upper()

df["Defect"] = df["Defect"].replace({
    "REP": "Repair",
    "REPAIR": "Repair",
    "REJ": "Reject",
    "REJECT": "Reject",
    "NAN": "None",
    "NONE":"None"
})

7. VIN format validation and masking for PII when exporting.

In [122]:
# clean VIN
df["VIN"] = df["VIN"].astype(str).str.strip().str.upper()

# keep only valid VINs (17 characters, no I O Q)
df = df[df["VIN"].str.match(r'^[A-HJ-NPR-Z0-9]{17}$')]

# mask VIN for export
df["VIN"] = df["VIN"].str[:3] + "***********" + df["VIN"].str[-3:]

8. Supplier code normalization and master join.

In [123]:
# normalize supplier code
df["Supplier"] = df["Supplier"].astype(str).str.strip().str.upper()

# extract numeric part
sup_num = df["Supplier"].str.extract(r'(\d+)')[0].astype(float)

# create standard code
df["Supplier"] = "SUP-" + sup_num.astype("Int64").astype(str).str.zfill(2)



9. Duplicate event dedup by (VIN, Station, TS).

In [124]:
# identify duplicates
df["Is_Duplicate"] = df.duplicated(subset=["VIN","Station","TS"], keep="first")

# count duplicates
df["Is_Duplicate"].sum()

#remove duplicates
df = df[~df["Is_Duplicate"]]

10. Sensor calibration factors and drift checks.

In [125]:
#Clean pressure column
df["Pressure"] = df["Pressure"].astype(str).str.strip().str.upper()

#Extract numeric value
pressure_val = df["Pressure"].str.extract(r'(\d+\.?\d*)')[0].astype(float)

#Detect PSI values
mask = df["Pressure"].str.contains("PSI", na=False)

#Convert psi to bar
pressure_val.loc[mask] = pressure_val.loc[mask] / 14.5038

#Replace original column
df["Pressure"] = pressure_val

In [126]:
#Apply calibration factor
factor = df["Calibration_Factor"].astype(float)

df["Torque"] = df["Torque"] * factor
df["Temp"] = df["Temp"] * factor
df["Pressure"] = df["Pressure"] * factor

In [127]:
#Detect torque drift
df = df.sort_values(["Machine_ID","TS"])

rolling_mean = df.groupby("Machine_ID")["Torque"].rolling(50).mean().reset_index(level=0, drop=True)

drift = rolling_mean.diff().abs() > 5

11. Cycle time plausibility (TS deltas) per station.

In [128]:
#Sort events
df = df.sort_values(["Station","TS"])
#Compute time difference
cycle_time = df.groupby("Station")["TS"].diff().dt.total_seconds()

#Detect Implausible Times
implausible = (cycle_time <= 0) | (cycle_time > 1800)

#removing 
df = df.loc[~implausible]
df = df.reset_index(drop=True)

12. Work order linkage validation (WO→VIN).

In [129]:
wo_counts = df.groupby("VIN")["WO"].nunique()
wo_counts[wo_counts > 1]

Series([], Name: WO, dtype: int64)

13. Rework loop identification and tagging.

In [130]:
#Detect repeated station visits
station_visits = df.groupby(["VIN","Station"]).size()
#Identify rework cases
rework = station_visits[station_visits > 1]
#Extract VINs with rework
rework_vins = rework.index.get_level_values("VIN").unique()

#Tagging rework
df["Rework_Flag"] = df["VIN"].isin(rework_vins)


14. Shift code derivation and validation.

In [131]:
hour = df["TS"].dt.hour
df["Shift_Derived"] = np.select(
    [
        (hour >= 6) & (hour < 14),
        (hour >= 14) & (hour < 22)
    ],
    ["Morning","Evening"],
    default="Night"
)


In [132]:
shift_mismatch = df["Shift"] != df["Shift_Derived"]
shift_mismatch.sum()

np.int64(2711)

In [133]:
df.loc[shift_mismatch, "Shift"] = df.loc[shift_mismatch, "Shift_Derived"]

In [134]:
(df["Shift"] != df["Shift_Derived"]).sum()

np.int64(0)

15. Scrap reason code mapping.

In [135]:
df['Scrap_Reason'].value_counts()

Scrap_Reason
Torque Out of Spec    835
Visual Defect         441
Temperature High      427
Leak Detected         425
Name: count, dtype: int64

In [136]:
df['Scrap_Reason'].isnull().sum()

np.int64(2053)

In [137]:
# Check distribution of scrap reasons
df['Scrap_Reason'].value_counts(dropna=False)

Scrap_Reason
NaN                   2053
Torque Out of Spec     835
Visual Defect          441
Temperature High       427
Leak Detected          425
Name: count, dtype: int64

In [138]:
# Replace missing scrap reasons with a placeholder
df['Scrap_Reason'] = df['Scrap_Reason'].fillna("Unknown")

In [139]:
# Standardize formatting
df['Scrap_Reason'] = (
    df['Scrap_Reason']
    .astype(str)
    .str.strip()
    .str.title()
)

In [140]:
df['Scrap_Reason'].value_counts(dropna=False)

Scrap_Reason
Unknown               2053
Torque Out Of Spec     835
Visual Defect          441
Temperature High       427
Leak Detected          425
Name: count, dtype: int64

16. Unit conversions for torque/temperature/pressure.


Clean Torque Values

In [141]:
# Remove text units and convert to numeric

df['Torque_Nm'] = (
    df['Torque']
    .astype(str)
    .str.replace("Nm", "", regex=False)
    .str.replace("nm", "", regex=False)
    .str.strip()
)

df['Torque_Nm'] = pd.to_numeric(df['Torque_Nm'], errors='coerce')

df[['Torque', 'Torque_Nm']].head()

,Torque,Torque_Nm
0,50.43766,50.43766
1,42.55680,42.55680
2,48.05904,48.05904
3,53.00760,53.00760
4,49.88251,49.88251


Convert Temperature to Celsius

In [142]:
def convert_temp(value):
    value = str(value).strip()
    
    try:
        if "F" in value:
            f = float(value.replace("F",""))
            return (f - 32) * 5/9
        elif "C" in value:
            return float(value.replace("C",""))
        else:
            return float(value)
    except:
        return None


df['Temp_C'] = df['Temp'].apply(convert_temp)

df[['Temp','Temp_C']].head()

,Temp,Temp_C
0,81.20658,81.20658
1,160.30720,160.30720
2,88.07430,88.07430
3,82.15200,82.15200
4,84.79035,84.79035


Convert Pressure to Bar

In [143]:
def convert_pressure(value):
    value = str(value).lower().strip()

    try:
        if "psi" in value:
            psi = float(value.replace("psi",""))
            return psi * 0.0689476
        elif "bar" in value:
            return float(value.replace("bar",""))
        else:
            return float(value)
    except:
        return None


df['Pressure_Bar'] = df['Pressure'].apply(convert_pressure)

df[['Pressure','Pressure_Bar']].head()

,Pressure,Pressure_Bar
0,9.83437,9.83437
1,9.32480,9.32480
2,10.08018,10.08018
3,9.48660,9.48660
4,9.61949,9.61949


In [144]:
print("Torque Missing:", df['Torque_Nm'].isnull().sum())
print("Temp Missing:", df['Temp_C'].isnull().sum())
print("Pressure Missing:", df['Pressure_Bar'].isnull().sum())

Torque Missing: 0
Temp Missing: 0
Pressure Missing: 0


17. Anomaly detection for out‑of‑spec readings.

In [145]:
# Define acceptable manufacturing ranges

TORQUE_MIN = 20
TORQUE_MAX = 80

TEMP_MIN = 10
TEMP_MAX = 120

PRESSURE_MIN = 1
PRESSURE_MAX = 15

CYCLE_MIN = 5
CYCLE_MAX = 300

In [146]:
# Convert Torque column to numeric Nm

df['Torque_Nm'] = (
    df['Torque']
    .astype(str)
    .str.replace('Nm', '', regex=False)
    .str.replace('nm', '', regex=False)
    .str.strip()
)

df['Torque_Nm'] = pd.to_numeric(df['Torque_Nm'], errors='coerce')

df[['Torque','Torque_Nm']].head()

,Torque,Torque_Nm
0,50.43766,50.43766
1,42.55680,42.55680
2,48.05904,48.05904
3,53.00760,53.00760
4,49.88251,49.88251


In [147]:
def convert_temp(value):
    value = str(value).strip()

    try:
        if "F" in value:
            f = float(value.replace("F",""))
            return (f - 32) * 5/9
        elif "C" in value:
            return float(value.replace("C",""))
        else:
            return float(value)
    except:
        return None


df['Temp_C'] = df['Temp'].apply(convert_temp)

In [148]:
def convert_pressure(value):
    value = str(value).lower().strip()

    try:
        if "psi" in value:
            psi = float(value.replace("psi",""))
            return psi * 0.0689476
        elif "bar" in value:
            return float(value.replace("bar",""))
        else:
            return float(value)
    except:
        return None


df['Pressure_Bar'] = df['Pressure'].apply(convert_pressure)

In [149]:
print(df[['Torque_Nm','Temp_C','Pressure_Bar']].head())

   Torque_Nm     Temp_C  Pressure_Bar
0   50.43766   81.20658       9.83437
1   42.55680  160.30720       9.32480
2   48.05904   88.07430      10.08018
3   53.00760   82.15200       9.48660
4   49.88251   84.79035       9.61949


In [150]:
# Torque anomaly detection

df['Torque_Anomaly'] = (
    (df['Torque_Nm'] < TORQUE_MIN) |
    (df['Torque_Nm'] > TORQUE_MAX)
)

df['Torque_Anomaly'].value_counts()

Torque_Anomaly
False    4158
True       23
Name: count, dtype: int64

In [151]:
df['Temp_Anomaly'] = (
    (df['Temp_C'] < TEMP_MIN) |
    (df['Temp_C'] > TEMP_MAX)
)

df['Temp_Anomaly'].value_counts()

Temp_Anomaly
False    3318
True      863
Name: count, dtype: int64

In [152]:
df['Pressure_Anomaly'] = (
    (df['Pressure_Bar'] < PRESSURE_MIN) |
    (df['Pressure_Bar'] > PRESSURE_MAX)
)

df['Pressure_Anomaly'].value_counts()

Pressure_Anomaly
False    4177
True        4
Name: count, dtype: int64

In [153]:
df['Cycle_Time'] = pd.to_numeric(df['Cycle_Time'], errors='coerce')

df['Cycle_Anomaly'] = (
    (df['Cycle_Time'] < CYCLE_MIN) |
    (df['Cycle_Time'] > CYCLE_MAX)
)

df['Cycle_Anomaly'].value_counts()

Cycle_Anomaly
False    4164
True       17
Name: count, dtype: int64

In [154]:
df['Sensor_Anomaly'] = (
    df['Torque_Anomaly'] |
    df['Temp_Anomaly'] |
    df['Pressure_Anomaly'] |
    df['Cycle_Anomaly']
)

df['Sensor_Anomaly'].value_counts()

Sensor_Anomaly
False    3283
True      898
Name: count, dtype: int64

Human vs automated inspection source tagging.

In [155]:
df['Inspection_Source'].value_counts(dropna=False)

Inspection_Source
human        731
AUTO         729
HUMAN        697
Human        677
automated    674
Automated    673
Name: count, dtype: int64

In [156]:
# Standardize inspection source values

df['Inspection_Source_Clean'] = (
    df['Inspection_Source']
    .astype(str)
    .str.strip()
    .str.lower()
)

inspection_map = {
    "human": "Human",
    "manual": "Human",
    
    "auto": "Automated",
    "automated": "Automated",
    "machine": "Automated"
}

df['Inspection_Source_Clean'] = df['Inspection_Source_Clean'].map(inspection_map)

In [157]:
df['Inspection_Source_Clean'] = df['Inspection_Source_Clean'].fillna("Unknown")
df['Inspection_Source_Clean'].value_counts()

Inspection_Source_Clean
Human        2105
Automated    2076
Name: count, dtype: int64

Tool ID Mapping for Torque Tools

In [158]:
df['Tool_ID'].value_counts()

Tool_ID
TL-016    180
TL-007    159
TL-006    159
TL-024    156
TL-013    155
TL-012    151
TL-019    148
TL-021    147
TL-027    146
TL-022    145
TL-020    144
TL-005    144
TL-004    142
TL-009    138
TL-002    137
TL-010    136
TL-017    136
TL-030    136
TL-011    135
TL-001    133
TL-008    132
TL-029    132
TL-028    131
TL-026    130
TL-023    128
TL-015    126
TL-003    124
TL-025    122
TL-018    116
TL-014    113
Name: count, dtype: int64

In [159]:
df['Tool_ID_Clean'] = (
    df['Tool_ID']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [160]:
df['Tool_Number'] = df['Tool_ID'].str.extract(r'(\d+)').astype(int)

In [161]:
df['Tool_Number'].max()

np.int64(30)

In [162]:
df['Tool_ID_Valid'] = df['Tool_ID_Clean'].str.match(r"^TL-\d{3}$")

In [163]:
df['Tool_ID_Valid'].value_counts()

Tool_ID_Valid
True    4181
Name: count, dtype: int64

In [164]:
tool_master = pd.DataFrame({
    "Tool_ID_Clean": [f"TL-{i:03d}" for i in range(1,31)],
    "Tool_Type": "Torque Wrench",
    "Calibration_Status": "Valid"
})

In [165]:
df = df.merge(tool_master, on="Tool_ID_Clean", how="left")

In [166]:
df['Tool_Master_Match'] = df['Tool_Type'].notna()
df['Tool_Master_Match'].value_counts()

Tool_Master_Match
True    4181
Name: count, dtype: int64

In [167]:
df['Tool_ID_Clean'].value_counts()

Tool_ID_Clean
TL-016    180
TL-007    159
TL-006    159
TL-024    156
TL-013    155
TL-012    151
TL-019    148
TL-021    147
TL-027    146
TL-022    145
TL-020    144
TL-005    144
TL-004    142
TL-009    138
TL-002    137
TL-010    136
TL-017    136
TL-030    136
TL-011    135
TL-001    133
TL-008    132
TL-029    132
TL-028    131
TL-026    130
TL-023    128
TL-015    126
TL-003    124
TL-025    122
TL-018    116
TL-014    113
Name: count, dtype: int64

BOM Version Effectivity Date Checks

In [168]:
df['BOM_Version'].unique()

<StringArray>
['BOM-3', 'BOM-2', 'BOM-1']
Length: 3, dtype: str

In [169]:
bom_master = pd.DataFrame({
    "BOM_Version": ["BOM-1", "BOM-2", "BOM-3"],
    "Effective_Date": [
        "2025-01-01",
        "2026-01-01",
        "2026-06-01"
    ]
})

bom_master["Effective_Date"] = pd.to_datetime(bom_master["Effective_Date"])

bom_master

,BOM_Version,Effective_Date
0,BOM-1,2025-01-01
1,BOM-2,2026-01-01
2,BOM-3,2026-06-01


In [170]:
df = df.merge(bom_master, on="BOM_Version", how="left")

In [171]:
df[['BOM_Version','Effective_Date']].head()

,BOM_Version,Effective_Date
0,BOM-3,2026-06-01
1,BOM-2,2026-01-01
2,BOM-2,2026-01-01
3,BOM-3,2026-06-01
4,BOM-3,2026-06-01


In [172]:
df[['BOM_Version','Effective_Date']].head()

,BOM_Version,Effective_Date
0,BOM-3,2026-06-01
1,BOM-2,2026-01-01
2,BOM-2,2026-01-01
3,BOM-3,2026-06-01
4,BOM-3,2026-06-01


In [176]:
df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

In [175]:
df['BOM_Valid'] = df['TS'] >= df['Effective_Date']